<h1>RAZ Systems </h1>

# Assignment -- AutoGen AgentChat: Hotel Reservation Assistant

### Problem Statement

You are tasked with building a simple hotel reservation lookup system using Python and SQLite. The system should allow front-desk staff to store and retrieve guest reservation details -- guest name, room type, and check-in date.

In addition to basic database operations, you will design a smart assistant agent that can answer staff questions about reservations in natural language.

This mirrors the lesson notebook's banking assistant, concept for concept -- just on a different dataset. **Your task:** fill in every `# TODO`. A full solution is at the end -- try not to peek until you've had a go.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### First concept: Select Which Model to use

**TODO:** create an `OpenAIChatCompletionClient` for `gpt-4o-mini`.

In [3]:
# --- TODO ---
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client =  OpenAIChatCompletionClient(model="gpt-4o-mini") # TODO: OpenAIChatCompletionClient(model="gpt-4o-mini")

### Second concept: Construct the Message

**TODO:** build a `TextMessage` asking about a specific reservation, with `source="staff"`.

In [5]:
# --- TODO ---
from autogen_agentchat.messages import TextMessage
message = TextMessage(
    content="What room type is reserved for confirmation number HTL2003?", 
    source="staff")  # TODO: a question about a reservation + who's asking
message

TextMessage(id='af8f8286-3205-4442-b10a-cc526b7cce55', source='staff', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 7, 10, 15, 36, 24, 466368, tzinfo=datetime.timezone.utc), content='What room type is reserved for confirmation number HTL2003?', type='TextMessage')

### Third concept: Build an Agent

**TODO:** create an `AssistantAgent` named `hotel_agent`, with a `system_message` describing a helpful, professional hotel front-desk assistant. Enable streaming.

In [ ]:
# --- TODO ---
from autogen_agentchat.agents import AssistantAgent

hotel_agent = AssistantAgent(
    name="hotel_Agent",             
    model_client=model_client,
    system_message=(
        "You are a helpful hotel front-desk assistant. "
        "You provide short, professional answers about guest reservations."
    ),
    model_client_stream=True,  # TODO
)

### Fourth concept: Talk to the agent with `on_messages` -- but it has no data yet

`on_messages()` is the AgentChat equivalent of `Runner.run(agent, message)` -- it takes a *list* of messages plus a `CancellationToken`, and returns a `Response` with `.chat_message`.

**TODO:** run the agent against your message and print the reply. Expect it to correctly refuse -- it has no real reservation data yet.

In [ ]:
# --- TODO ---
from autogen_core import CancellationToken

response = await hotel_agent.on_messages([message], cancellation_token=CancellationToken())  # TODO: agent + message
response.chat_message.content

### Fifth concept: Give the agent real data -- a local SQLite database of reservations

**TODO:** create a `hotel_reservations` table with columns `confirmation_number` (primary key), `guest_name`, `room_type`, `check_in_date`.

In [7]:
# --- TODO ---
import os
import sqlite3

if os.path.exists("hotel.db"):
    os.remove("hotel.db")

conn = sqlite3.connect("hotel.db")
c = conn.cursor()

c.execute("""
CREATE TABLE hotel_reservations (
    confirmation_number TEXT PRIMARY KEY,  
    guest_name TEXT,  
    room_type TEXT,  
    check_in_date TEXT   
)
""")

conn.commit()
conn.close()

In [9]:
# --- TODO: populate the database with sample reservations ---
def save_reservation(confirmation_number, guest_name, room_type, check_in_date):
    conn = sqlite3.connect("hotel.db")
    c = conn.cursor()
    c.execute("""
    REPLACE INTO hotel_reservations
    (confirmation_number, guest_name, room_type, check_in_date)
    VALUES (?, ?, ?, ?)
    """, (confirmation_number, guest_name, room_type, check_in_date))
    conn.commit()
    conn.close()


# TODO: add at least 5 sample reservations, e.g.
save_reservation("HTL2001", "Sarah Connor", "Deluxe King", "2026-07-01")
save_reservation("HTL2002", "John Doe", "Standard Twin", "2026-07-03")
save_reservation("HTL2003", "Priya Nair", "Executive Suite", "2026-07-05")
save_reservation("HTL2004", "Carlos Diaz", "Standard King", "2026-07-10")
save_reservation("HTL2005", "Mei Lin", "Deluxe Twin", "2026-07-12")

In [10]:
# --- TODO: write the lookup function ---
def get_reservation_details(confirmation_number: str) -> str:
    conn = sqlite3.connect("hotel.db")
    c = conn.cursor()
    c.execute("""
    SELECT guest_name, room_type, check_in_date
    FROM hotel_reservations
    WHERE confirmation_number = ?
    """, (confirmation_number,))
    result = c.fetchone()
    conn.close()

    if result:
        guest_name, room_type, check_in_date = result
        return f"Guest: {guest_name}, Room: {room_type}, Check-in: {check_in_date}"
    else:
        return "Reservation not found."

In [11]:
# Example usage -- TODO: try one of your own confirmation numbers
print(get_reservation_details("HTL2003"))  # TODO

Guest: Priya Nair, Room: Executive Suite, Check-in: 2026-07-05


### Sixth concept: Wire the database lookup in as a tool

**TODO:** create a new agent, `smart_hotel_agent`, with `get_reservation_details` passed directly in `tools=[...]`, and `reflect_on_tool_use=True` so the final answer is a natural sentence rather than the raw tool string.

In [14]:
# --- TODO ---
from autogen_agentchat.agents import AssistantAgent
smart_hotel_agent = AssistantAgent(
    name="smart_hotel_agent",                          # TODO
    model_client=model_client,
    system_message=(
        "You are a helpful hotel front-desk assistant. "
        "You provide short, professional answers about guest reservations."
    ),
    model_client_stream=True,
    tools=[get_reservation_details],                       # TODO: get_reservation_details
    reflect_on_tool_use=True,           # TODO
)

**TODO:** run the smart agent against your original message, print the inner messages (the `FunctionCall` + `FunctionExecutionResult`), then print the final reflected answer.

In [17]:
# --- TODO ---
from autogen_core import CancellationToken

response = await smart_hotel_agent.on_messages(
    [message], 
    cancellation_token=CancellationToken())  # TODO: smart_hotel_agent
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_FjFRpVOdgdeJtjNOD40gQo6K', arguments='{"confirmation_number":"HTL2003"}', name='get_reservation_details')]
[FunctionExecutionResult(content='Guest: Priya Nair, Room: Executive Suite, Check-in: 2026-07-05', name='get_reservation_details', call_id='call_FjFRpVOdgdeJtjNOD40gQo6K', is_error=False)]


'The reservation for confirmation number HTL2003 is for an Executive Suite, booked under the name Priya Nair. The check-in date is July 5, 2026.'

---
# Solution

No peeking until you've tried it yourself!

In [18]:
# === Setup ===
from dotenv import load_dotenv
load_dotenv(override=True)

from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [19]:
# === Message ===
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="What room type is reserved for confirmation number HTL2003?", source="staff")
message

TextMessage(id='79ca89b9-fa37-4645-878e-3fc0c3d84d74', source='staff', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 7, 10, 16, 26, 16, 3429, tzinfo=datetime.timezone.utc), content='What room type is reserved for confirmation number HTL2003?', type='TextMessage')

In [20]:
# === Plain agent, no tools yet ===
from autogen_agentchat.agents import AssistantAgent

hotel_agent = AssistantAgent(
    name="hotel_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful hotel front-desk assistant. "
        "You provide short, professional answers about guest reservations."
    ),
    model_client_stream=True,
)

In [21]:
from autogen_core import CancellationToken

response = await hotel_agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

"I'm sorry, but I can't access specific reservation details, including room types, by confirmation number. Please check our reservation system or contact customer support for assistance."

In [ ]:
# === Local database of reservations ===
import os
import sqlite3

if os.path.exists("hotel.db"):
    os.remove("hotel.db")

conn = sqlite3.connect("hotel.db")
c = conn.cursor()

c.execute("""
CREATE TABLE hotel_reservations (
    confirmation_number TEXT PRIMARY KEY,
    guest_name TEXT,
    room_type TEXT,
    check_in_date TEXT
)
""")

conn.commit()
conn.close()

In [ ]:
def save_reservation(confirmation_number, guest_name, room_type, check_in_date):
    conn = sqlite3.connect("hotel.db")
    c = conn.cursor()
    c.execute("""
    REPLACE INTO hotel_reservations
    (confirmation_number, guest_name, room_type, check_in_date)
    VALUES (?, ?, ?, ?)
    """, (confirmation_number, guest_name, room_type, check_in_date))
    conn.commit()
    conn.close()


save_reservation("HTL2001", "Sarah Connor", "Deluxe King", "2026-07-01")
save_reservation("HTL2002", "John Doe", "Standard Twin", "2026-07-03")
save_reservation("HTL2003", "Priya Nair", "Executive Suite", "2026-07-05")
save_reservation("HTL2004", "Carlos Diaz", "Standard King", "2026-07-10")
save_reservation("HTL2005", "Mei Lin", "Deluxe Twin", "2026-07-12")

In [ ]:
def get_reservation_details(confirmation_number: str) -> str:
    conn = sqlite3.connect("hotel.db")
    c = conn.cursor()
    c.execute("""
    SELECT guest_name, room_type, check_in_date
    FROM hotel_reservations
    WHERE confirmation_number = ?
    """, (confirmation_number,))
    result = c.fetchone()
    conn.close()

    if result:
        guest_name, room_type, check_in_date = result
        return f"Guest: {guest_name}, Room: {room_type}, Check-in: {check_in_date}"
    else:
        return "Reservation not found"


print(get_reservation_details("HTL2003"))

In [22]:
# === Smart agent, with the lookup wired in as a tool ===
smart_hotel_agent = AssistantAgent(
    name="smart_hotel_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful hotel front-desk assistant. "
        "You provide short, professional answers about guest reservations."
    ),
    model_client_stream=True,
    tools=[get_reservation_details],
    reflect_on_tool_use=True,
)

In [23]:
response = await smart_hotel_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_uTWvtHT8xxrH1khcludJTYRI', arguments='{"confirmation_number":"HTL2003"}', name='get_reservation_details')]
[FunctionExecutionResult(content='Guest: Priya Nair, Room: Executive Suite, Check-in: 2026-07-05', name='get_reservation_details', call_id='call_uTWvtHT8xxrH1khcludJTYRI', is_error=False)]


'The reservation for confirmation number HTL2003 is for an Executive Suite.'